In [1]:
import pandas as pd
import numpy as np

df = pd.concat([pd.read_csv("data/raw/fraudTrain.csv/fraudTrain.csv"),
                pd.read_csv("data/raw/fraudTest.csv/fraudTest.csv")], ignore_index=True)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df["ts"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values(["cc_num", "ts"]).reset_index(drop=True)
print(df.shape)

(1852394, 23)


In [2]:
g = df.groupby("cc_num")

# --- when ---
df["hour"] = df["ts"].dt.hour
df["is_night"] = df["hour"].isin([22, 23, 0, 1, 2, 3]).astype(int)
df["secs_since_prev"] = g["ts"].diff().dt.total_seconds().fillna(7 * 24 * 3600)

# --- how much, compared with THIS customer's past ---
prev_total = g["amt"].cumsum() - df["amt"]
prev_count = g.cumcount()
df["avg_prev_amt"] = (prev_total / prev_count.replace(0, np.nan)).fillna(df["amt"])
df["amt_vs_usual"] = df["amt"] / df["avg_prev_amt"]

# --- how fast: transactions in the last hour / 24 hours (excluding this one) ---
roll = df.set_index("ts").groupby("cc_num")["amt"]
df["txn_last_1h"] = roll.rolling("1h").count().to_numpy() - 1
df["txn_last_24h"] = roll.rolling("24h").count().to_numpy() - 1
df["spend_last_24h"] = roll.rolling("24h").sum().to_numpy() - df["amt"]

# --- where and with whom ---
df["new_merchant"] = (df.groupby(["cc_num", "merchant"]).cumcount() == 0).astype(int)
df["new_category"] = (df.groupby(["cc_num", "category"]).cumcount() == 0).astype(int)
lat1, lon1, lat2, lon2 = map(np.radians, [df["lat"], df["long"], df["merch_lat"], df["merch_long"]])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

# --- who ---
df["age"] = ((df["ts"] - pd.to_datetime(df["dob"])).dt.days / 365.25).round(1)

print("Features built ✅")
df[["amt", "amt_vs_usual", "txn_last_1h", "txn_last_24h", "secs_since_prev", "new_merchant", "distance_km", "age", "is_night"]].describe().round(2).T

Features built ✅


,count,mean,std,min,25%,50%,75%,max
amt,1852394.0,70.06,159.25,1.00,9.64,47.45,83.10,28948.90
amt_vs_usual,1852394.0,1.01,2.59,0.00,0.16,0.67,1.20,676.39
txn_last_1h,1852394.0,0.20,0.48,0.00,0.00,0.00,0.00,8.00
txn_last_24h,1852394.0,4.11,3.26,0.00,2.00,3.00,6.00,35.00
secs_since_prev,1852394.0,31248.96,47234.67,0.00,5701.00,15710.00,38194.00,1341471.00
new_merchant,1852394.0,0.29,0.45,0.00,0.00,0.00,1.00,1.00
distance_km,1852394.0,76.11,29.12,0.02,55.32,78.22,98.51,152.12
age,1852394.0,46.26,17.41,13.90,32.90,44.20,57.30,96.20
is_night,1852394.0,0.23,0.42,0.00,0.00,0.00,0.00,1.00


In [3]:
feats = ["amt", "amt_vs_usual", "txn_last_1h", "txn_last_24h", "spend_last_24h",
         "secs_since_prev", "new_merchant", "new_category", "distance_km", "age", "is_night"]
df.groupby("is_fraud")[feats].median().T.rename(columns={0: "genuine (median)", 1: "fraud (median)"}).round(2)

is_fraud,genuine (median),fraud (median)
amt,47.24,390.00
amt_vs_usual,0.67,5.24
txn_last_1h,0.00,0.00
txn_last_24h,3.00,4.00
spend_last_24h,182.50,1706.97
secs_since_prev,15756.00,5051.00
new_merchant,0.00,0.00
new_category,0.00,0.00
distance_km,78.22,78.10
age,44.20,47.80


In [4]:
df["period"] = np.select(
    [df["ts"] < "2020-01-01", df["ts"] < "2020-07-01"],
    ["train", "valid"], default="test")

keep = ["trans_num", "ts", "cc_num", "merchant", "category", "amt", "age", "gender", "state",
        "is_fraud", "period"] + [f for f in feats if f not in ["amt", "age"]] + ["hour"]
import os
os.makedirs("data/clean", exist_ok=True)
df[keep].to_parquet("data/clean/features.parquet", index=False)
print(df["period"].value_counts())
print("\nFraud rate by period (%):")
print((df.groupby("period")["is_fraud"].mean() * 100).round(3))

period
train    924850
test     525661
valid    401883
Name: count, dtype: int64

Fraud rate by period (%):
period
test     0.383
train    0.564
valid    0.602
Name: is_fraud, dtype: float64
